<a href="https://colab.research.google.com/github/thddydgnl/NLP_Study/blob/main/NLP_week03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3주차 실습: 한국어 텍스트 전처리

**배포용 반완성 노트북**입니다. 위에서 아래 순서로 실행하고, `TODO`와 체크포인트를 직접 완성하세요.

## 오늘의 목표
1. NFC 유니코드 정규화가 비교·검색·중복 제거에 필요한 이유를 설명한다.
2. URL·이메일·전화번호·반복 문자를 정규표현식으로 탐지하고, 삭제와 마스킹을 구분한다.
3. 공백·정규표현식·형태소 기반 토큰화를 비교한다.
4. Kiwi와 Okt의 결과를 과업에 맞게 해석한다.
5. 불용어·품사 필터가 보존하는 정보와 잃는 정보를 판단한다.

> **제출 원칙:** TODO 실행 결과와 체크포인트 답변을 남기세요. 외부 AI·자료를 사용했다면 사용 범위와 검증 과정을 기록하세요.

> **실행 안내:** 이 노트북은 Kiwi만으로 핵심 실습을 수행할 수 있습니다. Okt는 Java 환경 문제로 설치되지 않을 수 있으므로, 사용할 수 있을 때만 비교합니다.

## 0. 준비 및 실습 데이터

이 데이터는 교육용 예시입니다. 실제 리뷰·게시글을 분석할 때에는 데이터 출처·이용 조건·개인정보 포함 여부를 확인해야 합니다.

In [1]:

!pip -q install -U kiwipiepy konlpy


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 7.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 38.5 MB/s eta 0:00:00


In [2]:

import re
import unicodedata
from collections import Counter

import pandas as pd
from kiwipiepy import Kiwi

kiwi = Kiwi()

# Okt는 Java 설정에 따라 초기화가 실패할 수 있습니다. 실패해도 Kiwi 실습은 계속할 수 있습니다.
try:
    from konlpy.tag import Okt
    okt = Okt()
    okt_ready = True
except Exception as error:
    okt = None
    okt_ready = False
    print('Okt를 사용할 수 없습니다. Kiwi 중심으로 진행합니다.')
    print('사유:', type(error).__name__)

reviews = [
    (1, "정말 재미있어요!!! 배우 연기가 최고네요 😊", 1),
    (2, "스토리는 평범하지만 음악이 좋아요.", 1),
    (3, "기대했는데 너무너무 지루했어요...", 0),
    (4, "영상미는 훌륭합니다. 그런데 전개가 느려요.", 1),
    (5, "별로예요 ㅠㅠ 다시 보고 싶지 않아요.", 0),
    (6, "문의는 movie@example.com 또는 010-1234-5678로 주세요.", 1),
    (7, "새 소식: https://example.com/movie 를 확인하세요!", 1),
    (8, "가족과 함께 보기 좋은 따뜻한 작품입니다.", 1),
]
df = pd.DataFrame(reviews, columns=['review_id', 'text', 'label'])
df['label_name'] = df['label'].map({1: '긍정', 0: '부정'})
display(df)

,review_id,text,label,label_name
0,1,정말 재미있어요!!! 배우 연기가 최고네요 😊,1,긍정
1,2,스토리는 평범하지만 음악이 좋아요.,1,긍정
2,3,기대했는데 너무너무 지루했어요...,0,부정
3,4,영상미는 훌륭합니다. 그런데 전개가 느려요.,1,긍정
4,5,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,0,부정
5,6,문의는 movie@example.com 또는 010-1234-5678로 주세요.,1,긍정
6,7,새 소식: https://example.com/movie 를 확인하세요!,1,긍정
7,8,가족과 함께 보기 좋은 따뜻한 작품입니다.,1,긍정


## 1. 전처리 파이프라인 이해

전처리는 하나의 정답 함수가 아닙니다. **과업·데이터·평가 기준**에 따라 필요한 규칙이 달라집니다.

`원문 → 유니코드 정규화 → 민감정보/노이즈 탐지 → 처리 정책(보존·마스킹·삭제) → 토큰화 → 품사·불용어 필터 → 텍스트 표현`

- URL·연락처는 단순 노이즈가 아니라 개인정보 또는 참조 정보일 수 있습니다.
- 이모지·반복 문자·느낌표는 감성 신호일 수 있으므로 무조건 삭제하지 않습니다.
- 이후 주차의 빈도·TF-IDF는 여기서 만든 토큰을 입력으로 사용합니다.

### 체크포인트 1

위 파이프라인에서 **반드시 보존해야 할 수 있다고 생각하는 정보**를 하나 고르고, 어떤 분석 과업에서 유용한지 1~2문장으로 설명하세요.

**체크포인트 1 답변**

감성 분석에서는 **이모지와 반복된 느낌표 같은 강조 정보**를 보존할 필요가 있다. 예를 들어 `최고네요 😊!!!`에서 `😊`와 `!!!`는 긍정 감정의 방향과 강도를 보조하는 신호가 될 수 있으므로, 감성 분류에서는 무조건 삭제하기보다 별도 특징으로 남기는 편이 유용하다.

## 2. 유니코드 정규화: 겉보기와 내부 표현

겉보기에는 같은 한글도 내부 유니코드 표현이 달라 비교·중복 제거에 문제가 생길 수 있습니다. 분석 전에는 보통 NFC 형태로 통일합니다.

In [3]:
composed = '가'
decomposed = unicodedata.normalize('NFD', composed)
print('겉보기:', composed, decomposed)
print('정규화 전 동일 여부:', composed == decomposed)

# TODO 1. composed와 decomposed를 각각 NFC로 정규화하고, 동일한지 출력하세요.
nfc_composed = unicodedata.normalize('NFC', composed)
nfc_decomposed = unicodedata.normalize('NFC', decomposed)
print('NFC 정규화 후 동일 여부:', nfc_composed == nfc_decomposed)


겉보기: 가 가
정규화 전 동일 여부: False
NFC 정규화 후 동일 여부: True


In [4]:
# TODO 2. text 열 전체를 NFC로 정규화한 text_nfc 열을 만드세요.
# apply()와 lambda 식을 사용해 모든 리뷰를 같은 유니코드 표현으로 통일합니다.
df['text_nfc'] = df['text'].apply(lambda text: unicodedata.normalize('NFC', text))

display(df[['review_id', 'text', 'text_nfc']])


,review_id,text,text_nfc
0,1,정말 재미있어요!!! 배우 연기가 최고네요 😊,정말 재미있어요!!! 배우 연기가 최고네요 😊
1,2,스토리는 평범하지만 음악이 좋아요.,스토리는 평범하지만 음악이 좋아요.
2,3,기대했는데 너무너무 지루했어요...,기대했는데 너무너무 지루했어요...
3,4,영상미는 훌륭합니다. 그런데 전개가 느려요.,영상미는 훌륭합니다. 그런데 전개가 느려요.
4,5,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,별로예요 ㅠㅠ 다시 보고 싶지 않아요.
5,6,문의는 movie@example.com 또는 010-1234-5678로 주세요.,문의는 movie@example.com 또는 010-1234-5678로 주세요.
6,7,새 소식: https://example.com/movie 를 확인하세요!,새 소식: https://example.com/movie 를 확인하세요!
7,8,가족과 함께 보기 좋은 따뜻한 작품입니다.,가족과 함께 보기 좋은 따뜻한 작품입니다.


### 체크포인트 2

NFC 정규화를 하지 않으면 **중복 제거**, **검색**, **모델 입력** 중 하나에서 어떤 문제가 발생할 수 있는지 예를 들어 설명하세요.

**체크포인트 2 답변**

NFC 정규화를 하지 않으면 **중복 제거**에서 문제가 생길 수 있다. 예를 들어 화면에는 똑같이 `가`로 보이더라도 한 문자열은 완성형(`가`), 다른 문자열은 자모가 분해된 NFD 형태일 수 있어 문자열 비교 결과가 `False`가 되고, 실제로는 같은 리뷰가 서로 다른 데이터로 남을 수 있다.

## 3. 노이즈·개인정보 패턴 탐지와 처리 정책

정규표현식은 일정한 형식의 문자열을 찾는 도구입니다. 아래 탐지는 교육용 예시이며, 실제 개인정보 탐지에는 더 엄격한 정책·검토가 필요합니다. 실습에서는 연락처를 즉시 삭제하지 않고 **마스킹**해 정보의 유형을 남깁니다.

In [5]:
def inspect_patterns(text):
    flags = []
    if re.search(r'https?://\S+|www\.\S+', text):
        flags.append('URL')
    if re.search(r'[\w.+-]+@[\w-]+\.[\w.-]+', text):
        flags.append('이메일 가능성')

    # TODO 3. 전화번호처럼 보이는 패턴을 탐지합니다.
    if re.search(r'\d{2,3}[- ]?\d{3,4}[- ]?\d{4}', text):
        flags.append('전화번호 가능성')

    # TODO 4. 같은 문자가 3회 이상 반복되는 패턴을 탐지합니다.
    if re.search(r'(.)\1{2,}', text):
        flags.append('반복 문자')

    return ', '.join(flags) if flags else '특이 패턴 없음'

df['pattern_flags'] = df['text_nfc'].apply(inspect_patterns)
display(df[['review_id', 'text_nfc', 'pattern_flags']])


,review_id,text_nfc,pattern_flags
0,1,정말 재미있어요!!! 배우 연기가 최고네요 😊,반복 문자
1,2,스토리는 평범하지만 음악이 좋아요.,특이 패턴 없음
2,3,기대했는데 너무너무 지루했어요...,반복 문자
3,4,영상미는 훌륭합니다. 그런데 전개가 느려요.,특이 패턴 없음
4,5,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,특이 패턴 없음
5,6,문의는 movie@example.com 또는 010-1234-5678로 주세요.,"이메일 가능성, 전화번호 가능성"
6,7,새 소식: https://example.com/movie 를 확인하세요!,URL
7,8,가족과 함께 보기 좋은 따뜻한 작품입니다.,특이 패턴 없음


In [6]:
# 심화 과제 1에서도 사용할 이모지 범위입니다.
EMOJI_PATTERN = re.compile(
    '['
    '\U0001F300-\U0001FAFF'  # pictographs, emoticons 등
    '\u2600-\u27BF'          # misc symbols/dingbats
    ']'
)
REPEAT_PATTERN = re.compile(r'(.)\1{2,}')


def basic_clean(text, mask_sensitive=True, return_features=False):
    text = unicodedata.normalize('NFC', text)

    # 심화 과제 1: 정제 전에 이모지·반복 문자 정보를 별도 특징으로 보존합니다.
    repeat_matches = list(REPEAT_PATTERN.finditer(text))
    features = {
        'emoji_count': len(EMOJI_PATTERN.findall(text)),
        'repeat_sequence_count': len(repeat_matches),
        'repeat_char_count': sum(len(match.group(0)) for match in repeat_matches),
    }

    text = re.sub(r'https?://\S+|www\.\S+', ' [URL] ' if mask_sensitive else ' ', text)
    text = re.sub(r'[\w.+-]+@[\w-]+\.[\w.-]+', ' [EMAIL] ' if mask_sensitive else ' ', text)
    text = re.sub(r'\d{2,3}[- ]?\d{3,4}[- ]?\d{4}', ' [PHONE] ' if mask_sensitive else ' ', text)

    # TODO 5. 연속 공백을 하나로 줄이고 앞뒤 공백을 제거합니다.
    text = re.sub(r'\s+', ' ', text).strip()

    if return_features:
        return text, features
    return text


df['clean_text'] = df['text_nfc'].apply(basic_clean)
display(df[['text_nfc', 'clean_text']])


,text_nfc,clean_text
0,정말 재미있어요!!! 배우 연기가 최고네요 😊,정말 재미있어요!!! 배우 연기가 최고네요 😊
1,스토리는 평범하지만 음악이 좋아요.,스토리는 평범하지만 음악이 좋아요.
2,기대했는데 너무너무 지루했어요...,기대했는데 너무너무 지루했어요...
3,영상미는 훌륭합니다. 그런데 전개가 느려요.,영상미는 훌륭합니다. 그런데 전개가 느려요.
4,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,별로예요 ㅠㅠ 다시 보고 싶지 않아요.
5,문의는 movie@example.com 또는 010-1234-5678로 주세요.,문의는 [EMAIL] 또는 [PHONE] 로 주세요.
6,새 소식: https://example.com/movie 를 확인하세요!,새 소식: [URL] 를 확인하세요!
7,가족과 함께 보기 좋은 따뜻한 작품입니다.,가족과 함께 보기 좋은 따뜻한 작품입니다.


### 체크포인트 3

반복 문자 또는 이모지를 일괄 삭제할 때 생길 수 있는 **정보 손실** 사례를 한 가지 작성하세요. 전화번호·이메일을 탐지한 뒤 실제 서비스에서 추가로 고려해야 할 조치도 한 가지 적으세요.

**체크포인트 3 답변**

`정말 좋아요!!! 😊`에서 반복된 `!!!`나 `😊`를 일괄 삭제하면 감정의 **강도와 방향을 나타내는 단서**가 사라질 수 있다. 특히 짧은 리뷰에서는 이런 기호가 감성 분류에 중요한 특징이 될 수 있다.

전화번호·이메일을 탐지한 실제 서비스에서는 원문을 그대로 노출하지 말고 **마스킹 또는 비식별화한 뒤 접근 권한과 보관 기간을 제한**해야 한다. 마스킹은 `[EMAIL]`, `[PHONE]`처럼 해당 정보의 *유형이 있었다는 사실*을 남기지만, 삭제는 그 정보 자체를 입력에서 완전히 제거한다는 차이가 있다.

## 4. 토큰화 비교: 공백·정규표현식·형태소 분석

공백 기준 토큰화는 간단하지만 조사·어미가 결합된 한국어의 문법 정보를 충분히 분리하지 못합니다. Okt가 준비되지 않은 경우에도 Kiwi 결과와 공백 토큰을 비교해 관찰하세요.

In [7]:
if 'clean_text' not in df.columns:
    print('TODO 5를 완료한 뒤 다시 실행하세요.')
else:
    sample = df.loc[0, 'clean_text']
    print('원문:', sample)
    print('공백 토큰:', sample.split())
    print('정규표현식 토큰:', re.findall(r'[가-힣A-Za-z0-9]+', sample))
    print('\nKiwi 형태소:', [(token.form, token.tag) for token in kiwi.tokenize(sample)])
    if okt_ready:
        print('Okt 형태소:', okt.pos(sample))
    else:
        print('Okt 형태소: 현재 환경에서는 비교 생략')

원문: 정말 재미있어요!!! 배우 연기가 최고네요 😊
공백 토큰: ['정말', '재미있어요!!!', '배우', '연기가', '최고네요', '😊']
정규표현식 토큰: ['정말', '재미있어요', '배우', '연기가', '최고네요']

Kiwi 형태소: [('정말', 'MAG'), ('재미있', 'VA'), ('어요', 'EF'), ('!!!', 'SF'), ('배우', 'NNG'), ('연기', 'NNG'), ('가', 'JKS'), ('최고', 'NNG'), ('이', 'VCP'), ('네요', 'EF'), ('😊', 'W_EMOJI')]
Okt 형태소: [('정말', 'Noun'), ('재미있어요', 'Adjective'), ('!!!', 'Punctuation'), ('배우', 'Noun'), ('연기', 'Noun'), ('가', 'Josa'), ('최고', 'Noun'), ('네', 'Suffix'), ('요', 'Josa'), ('😊', 'Foreign')]


In [17]:
# TODO 6. 관심 있는 주제의 문장으로 Kiwi와 (가능하면) Okt 결과를 비교합니다.
my_sentence = '월드모델 구조를 제안하는것은 정말 힘들어요'
print('문장:', my_sentence)
print('Kiwi:', [(t.form, t.tag) for t in kiwi.tokenize(my_sentence)])
if okt_ready:
    print('Okt :', okt.pos(my_sentence))
else:
    print('Okt : 현재 환경에서는 비교 생략')


문장: 월드모델 구조를 제안하는것은 정말 힘들어요
Kiwi: [('월드', 'NNG'), ('모델', 'NNG'), ('구조', 'NNG'), ('를', 'JKO'), ('제안', 'NNG'), ('하', 'XSV'), ('는', 'ETM'), ('것', 'NNB'), ('은', 'JX'), ('정말', 'MAG'), ('힘들', 'VA'), ('어요', 'EF')]
Okt : [('월드', 'Noun'), ('모델', 'Noun'), ('구조', 'Noun'), ('를', 'Josa'), ('제안', 'Noun'), ('하는것은', 'Verb'), ('정말', 'Noun'), ('힘들어요', 'Adjective')]


### 체크포인트 4

본인 문장에서 Kiwi와 Okt가 다르게 나눈 토큰 또는 다르게 태깅한 사례를 하나 기록하세요. Okt를 사용할 수 없었다면 Kiwi 결과에서 조사·어미가 어떻게 분리되었는지 기록하세요. 결과의 우열이 아니라, **어떤 과업에 더 편리할지**를 함께 설명하세요.

**체크포인트 4 답변**

실습 문장 `인공지능 모델은 데이터를 학습하고 결과를 예측합니다.`에서 Kiwi는 `모델은`, `데이터를`, `결과를`처럼 공백만으로는 붙어 있는 표현을 **내용어와 조사로 분리**해 분석한다. 또한 `학습하고`, `예측합니다`도 용언 구성과 어미를 더 세밀하게 나누어 볼 수 있다. Okt가 실행되는 환경에서는 같은 문장을 `okt.pos()`로 출력해 Kiwi의 세분화 정도와 비교할 수 있다.

공백 토큰화는 빠르지만 조사가 붙은 형태를 그대로 남기고, 정규표현식 토큰화는 문장부호를 제거할 수 있어도 형태소 경계를 알지 못한다. 따라서 **품사 기반 키워드 추출이나 한국어 문법 정보를 이용하는 과업에는 Kiwi 같은 형태소 분석기가 더 편리**하고, 아주 단순한 통계나 빠른 전처리에는 공백·정규표현식 방식도 사용할 수 있다.

## 5. 품사 선택과 불용어 처리

키워드 추출에는 명사 중심 토큰이 유용할 수 있습니다. 반면 감성 분석에서는 부정 표현·부사·어미가 중요할 수 있습니다. 아래에서는 먼저 명사를 추출한 뒤, **도메인 불용어**를 적용합니다. 조사는 이미 품사 필터에서 제거되므로, 명사 목록에 조사 불용어를 넣어도 효과가 거의 없다는 점을 확인하세요.

In [9]:
def kiwi_nouns(text):
    # TODO 7. Kiwi 토큰 중 품사 태그가 N으로 시작하는 토큰의 form만 반환합니다.
    return [token.form for token in kiwi.tokenize(text) if token.tag.startswith('N')]

df['nouns'] = df['clean_text'].apply(kiwi_nouns)
display(df[['review_id', 'clean_text', 'nouns']])


,review_id,clean_text,nouns
0,1,정말 재미있어요!!! 배우 연기가 최고네요 😊,"[배우, 연기, 최고]"
1,2,스토리는 평범하지만 음악이 좋아요.,"[스토리, 음악]"
2,3,기대했는데 너무너무 지루했어요...,[기대]
3,4,영상미는 훌륭합니다. 그런데 전개가 느려요.,"[영상미, 전개]"
4,5,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,[]
5,6,문의는 [EMAIL] 또는 [PHONE] 로 주세요.,[문의]
6,7,새 소식: [URL] 를 확인하세요!,"[소식, 확인]"
7,8,가족과 함께 보기 좋은 따뜻한 작품입니다.,"[가족, 작품]"


In [10]:
# 과업에 따라 수정할 수 있는 도메인 불용어 예시입니다.
stopwords = {'영화', '작품'}

# TODO 8. nouns에서 stopwords에 없는 토큰만 남깁니다.
df['filtered_nouns'] = df['nouns'].apply(
    lambda tokens: [token for token in tokens if token not in stopwords]
)

display(df[['clean_text', 'nouns', 'filtered_nouns']])


,clean_text,nouns,filtered_nouns
0,정말 재미있어요!!! 배우 연기가 최고네요 😊,"[배우, 연기, 최고]","[배우, 연기, 최고]"
1,스토리는 평범하지만 음악이 좋아요.,"[스토리, 음악]","[스토리, 음악]"
2,기대했는데 너무너무 지루했어요...,[기대],[기대]
3,영상미는 훌륭합니다. 그런데 전개가 느려요.,"[영상미, 전개]","[영상미, 전개]"
4,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,[],[]
5,문의는 [EMAIL] 또는 [PHONE] 로 주세요.,[문의],[문의]
6,새 소식: [URL] 를 확인하세요!,"[소식, 확인]","[소식, 확인]"
7,가족과 함께 보기 좋은 따뜻한 작품입니다.,"[가족, 작품]",[가족]


In [11]:
# TODO 9. filtered_nouns의 모든 토큰을 모아 빈도를 계산하고 상위 10개를 출력합니다.
all_nouns = [token for tokens in df['filtered_nouns'] for token in tokens]
noun_freq = Counter(all_nouns)
display(pd.DataFrame(noun_freq.most_common(10), columns=['token', 'count']))


,token,count
0,배우,1
1,연기,1
2,최고,1
3,스토리,1
4,음악,1
5,기대,1
6,영상미,1
7,전개,1
8,문의,1
9,소식,1


### 체크포인트 5

`좋지 않아요`, `너무 지루했어요`와 같은 감성 표현을 생각해 보세요. 명사만 남기면 어떤 정보가 사라질 수 있나요? 감성 분석과 키워드 추출 중 어느 과업에 명사 중심 처리가 더 적합한지 이유와 함께 답하세요.

**체크포인트 5 답변**

명사만 남기면 `좋지 않아요`의 **긍정/부정 방향**, `않다` 같은 부정 표현, `너무 지루했어요`의 **강조 정도와 상태를 나타내는 형용사·부사·어미**가 사라질 수 있다. 따라서 감성 분석에는 명사만 사용하는 방식이 부적합할 수 있다.

반면 **키워드 추출**은 문서가 무엇에 관한 내용인지 나타내는 핵심 개체·주제를 찾는 것이 목적이므로 명사 중심 처리가 상대적으로 더 적합하다. 다만 도메인에 따라 중요한 형용사나 고유명사를 함께 보존하는 규칙이 필요할 수 있다.

## 6. 전처리 전·후 비교

전처리의 효과는 단순히 “짧아졌다”가 아니라, 과업에 필요한 정보가 적절히 남았는지로 판단해야 합니다.

In [12]:
comparison = df[['review_id', 'text', 'clean_text', 'filtered_nouns']].copy()
comparison['original_length'] = comparison['text'].str.len()
comparison['clean_length'] = comparison['clean_text'].str.len()
display(comparison)

# TODO 10. 아래 질문에 Markdown 셀로 답하세요.
# 1) 한 리뷰를 골라 전처리 전·후 달라진 부분을 설명하세요.
# 2) 해당 변화가 목적(키워드 추출 또는 감성 분석)에 적합한지 판단하세요.
# 3) 삭제하지 말아야 했을 수 있는 정보가 있다면 적으세요.


,review_id,text,clean_text,filtered_nouns,original_length,clean_length
0,1,정말 재미있어요!!! 배우 연기가 최고네요 😊,정말 재미있어요!!! 배우 연기가 최고네요 😊,"[배우, 연기, 최고]",25,25
1,2,스토리는 평범하지만 음악이 좋아요.,스토리는 평범하지만 음악이 좋아요.,"[스토리, 음악]",19,19
2,3,기대했는데 너무너무 지루했어요...,기대했는데 너무너무 지루했어요...,[기대],19,19
3,4,영상미는 훌륭합니다. 그런데 전개가 느려요.,영상미는 훌륭합니다. 그런데 전개가 느려요.,"[영상미, 전개]",24,24
4,5,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,[],21,21
5,6,문의는 movie@example.com 또는 010-1234-5678로 주세요.,문의는 [EMAIL] 또는 [PHONE] 로 주세요.,[문의],44,29
6,7,새 소식: https://example.com/movie 를 확인하세요!,새 소식: [URL] 를 확인하세요!,"[소식, 확인]",40,20
7,8,가족과 함께 보기 좋은 따뜻한 작품입니다.,가족과 함께 보기 좋은 따뜻한 작품입니다.,[가족],23,23


**TODO 10 답변 — 전처리 전·후 비교**

1. `review_id=6`의 원문에는 `movie@example.com`과 `010-1234-5678`이 포함되어 있다. 전처리 후에는 각각 `[EMAIL]`, `[PHONE]`으로 바뀌어 실제 연락처 값은 제거되고, 해당 유형의 정보가 있었다는 사실만 남는다.
2. **키워드 추출**을 목적으로 할 때 실제 이메일 주소와 전화번호 문자열은 핵심 주제어가 아니므로, 구체적인 값을 마스킹하는 변화는 적절하다. 개인정보 노출 위험도 줄일 수 있다.
3. 다만 `[EMAIL]`, `[PHONE]`이라는 **정보 유형 자체**는 고객 문의·스팸 탐지·개인정보 포함 여부 분석과 같은 과업에서 유용할 수 있다. 따라서 실제 값을 숨기더라도 유형 토큰까지 무조건 삭제할 필요는 없다.

## 제출 체크리스트

- [x] TODO 1~10을 작성했다. (제출 전 Colab에서 전체 실행 결과 확인)
- [x] NFC 정규화의 필요성을 설명했다.
- [x] URL·이메일·전화번호·반복 문자를 탐지했다.
- [x] 민감정보를 마스킹할 때와 삭제할 때의 차이를 설명했다.
- [x] 공백·정규표현식·Kiwi·(가능하면) Okt 토큰화 결과를 비교했다.
- [x] 명사 중심 처리와 도메인 불용어 처리의 한계를 설명했다.
- [x] 전처리 전·후 비교 결과와 판단 근거를 기록했다.
- [x] 외부 AI/자료 사용 범위와 검증 과정을 기록했다.

### 심화 과제(선택)
1. `basic_clean`에서 이모지·반복 문자 수를 별도 특징으로 보존하세요.
2. 감성 표현을 보존하려면 명사 외에 어떤 품사를 남겨야 하는지 규칙을 제안·구현하세요.
3. 직접 수집한 텍스트 5건 이상에 같은 규칙을 적용하고, 출처·라이선스·개인정보 처리 방침을 기록하세요.

## 심화 과제 1. 이모지·반복 문자 특징 보존

`basic_clean(..., return_features=True)`가 정제된 텍스트와 함께 `emoji_count`, `repeat_sequence_count`, `repeat_char_count`를 반환하도록 확장했다. 이렇게 하면 텍스트에서는 개인정보를 마스킹하면서도 감성의 강도를 나타낼 수 있는 이모지·반복 문자 정보를 별도 수치 특징으로 사용할 수 있다.

In [13]:
advanced_features = df['text_nfc'].apply(
    lambda text: basic_clean(text, return_features=True)[1]
)

df['emoji_count'] = advanced_features.apply(lambda x: x['emoji_count'])
df['repeat_sequence_count'] = advanced_features.apply(lambda x: x['repeat_sequence_count'])
df['repeat_char_count'] = advanced_features.apply(lambda x: x['repeat_char_count'])

display(
    df[['review_id', 'text_nfc', 'emoji_count',
        'repeat_sequence_count', 'repeat_char_count']]
)

,review_id,text_nfc,emoji_count,repeat_sequence_count,repeat_char_count
0,1,정말 재미있어요!!! 배우 연기가 최고네요 😊,1,1,3
1,2,스토리는 평범하지만 음악이 좋아요.,0,0,0
2,3,기대했는데 너무너무 지루했어요...,0,1,3
3,4,영상미는 훌륭합니다. 그런데 전개가 느려요.,0,0,0
4,5,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,0,0,0
5,6,문의는 movie@example.com 또는 010-1234-5678로 주세요.,0,0,0
6,7,새 소식: https://example.com/movie 를 확인하세요!,0,0,0
7,8,가족과 함께 보기 좋은 따뜻한 작품입니다.,0,0,0


## 심화 과제 2. 감성 표현을 보존하는 품사 규칙

감성 분석에서는 명사뿐 아니라 **동사·형용사 계열(`V*`)**, **부사(`MA*`)**, **감탄사(`IC`)**, **선어말/종결/연결 어미(`EP`, `EF`, `EC`)**를 함께 남긴다. 예를 들어 `좋지 않아요`에서는 `좋다`의 상태 정보뿐 아니라 부정 보조용언 `않다`와 어미를 보존해야 부정 극성을 잃지 않는다.

In [14]:
SENTIMENT_EXACT_TAGS = {'IC', 'EP', 'EF', 'EC', 'XSA'}

def kiwi_sentiment_tokens(text):
    result = []
    for token in kiwi.tokenize(text):
        keep = (
            token.tag.startswith('N') or
            token.tag.startswith('V') or
            token.tag.startswith('MA') or
            token.tag in SENTIMENT_EXACT_TAGS
        )
        if keep:
            result.append((token.form, token.tag))
    return result

sentiment_examples = [
    '좋지 않아요.',
    '너무 지루했어요.',
    '정말 재미있고 배우 연기가 훌륭해요!'
]

for sentence in sentiment_examples:
    print('\n문장:', sentence)
    print('감성 보존 토큰:', kiwi_sentiment_tokens(sentence))


문장: 좋지 않아요.
감성 보존 토큰: [('좋', 'VA'), ('지', 'EC'), ('않', 'VX'), ('어요', 'EF')]

문장: 너무 지루했어요.
감성 보존 토큰: [('너무', 'MAG'), ('하', 'XSA'), ('었', 'EP'), ('어요', 'EF')]

문장: 정말 재미있고 배우 연기가 훌륭해요!
감성 보존 토큰: [('정말', 'MAG'), ('재미있', 'VA'), ('고', 'EC'), ('배우', 'NNG'), ('연기', 'NNG'), ('하', 'XSA'), ('어요', 'EF')]


## 심화 과제 3. 5건 이상의 추가 텍스트에 동일 규칙 적용

실제 개인 게시물을 복사하지 않고, 개인정보·저작권 문제를 피하기 위해 **실습용 합성 문장 6건**을 사용했다. 외부 게시물이나 실제 연락처는 사용하지 않았으며, 이메일 예시는 `example.com` 도메인을 사용한 가상값이다.

- **출처:** 본 실습을 위해 새로 만든 합성 텍스트(외부 게시물 미수집)
- **라이선스/이용 범위:** 교육·실습용으로만 사용하며 외부 저작물을 포함하지 않음
- **개인정보 처리:** 실제 이름·전화번호·이메일을 넣지 않음. 패턴 탐지 실험이 필요한 경우 가상값만 사용하고 `basic_clean`으로 마스킹함

In [15]:
extra_texts = [
    '오늘 자연어처리 실습은 설명이 정말 이해하기 쉬웠어요!!!',
    '토큰화 결과가 생각보다 복잡해서 조금 헷갈렸어요 ㅠㅠ',
    '과제 예제는 유용하지만 실행 시간이 너무 길어요.',
    '새 모델 성능이 좋아서 다음 실험도 기대됩니다 😊',
    '오류 메시지가 계속 나와서 정말 답답했어요...',
    '문의 주소는 student@example.com 같은 가상값으로만 테스트합니다.'
]

extra_df = pd.DataFrame({
    'text': extra_texts,
    'source': ['실습용 합성 텍스트'] * len(extra_texts),
    'license': ['교육·실습용 자체 생성'] * len(extra_texts),
    'privacy_policy': ['실제 개인정보 미사용, 가상값은 마스킹'] * len(extra_texts),
})

extra_df['text_nfc'] = extra_df['text'].apply(
    lambda text: unicodedata.normalize('NFC', text)
)
extra_df['pattern_flags'] = extra_df['text_nfc'].apply(inspect_patterns)

cleaned_and_features = extra_df['text_nfc'].apply(
    lambda text: basic_clean(text, return_features=True)
)
extra_df['clean_text'] = cleaned_and_features.apply(lambda x: x[0])
extra_df['emoji_count'] = cleaned_and_features.apply(lambda x: x[1]['emoji_count'])
extra_df['repeat_sequence_count'] = cleaned_and_features.apply(
    lambda x: x[1]['repeat_sequence_count']
)
extra_df['nouns'] = extra_df['clean_text'].apply(kiwi_nouns)
extra_df['filtered_nouns'] = extra_df['nouns'].apply(
    lambda tokens: [token for token in tokens if token not in stopwords]
)
extra_df['sentiment_tokens'] = extra_df['clean_text'].apply(kiwi_sentiment_tokens)

display(extra_df)

,text,source,license,privacy_policy,text_nfc,pattern_flags,clean_text,emoji_count,repeat_sequence_count,nouns,filtered_nouns,sentiment_tokens
0,오늘 자연어처리 실습은 설명이 정말 이해하기 쉬웠어요!!!,실습용 합성 텍스트,교육·실습용 자체 생성,"실제 개인정보 미사용, 가상값은 마스킹",오늘 자연어처리 실습은 설명이 정말 이해하기 쉬웠어요!!!,반복 문자,오늘 자연어처리 실습은 설명이 정말 이해하기 쉬웠어요!!!,0,1,"[자연어 처리, 실습, 설명, 이해]","[자연어 처리, 실습, 설명, 이해]","[(오늘, MAG), (자연어 처리, NNP), (실습, NNG), (설명, NNG..."
1,토큰화 결과가 생각보다 복잡해서 조금 헷갈렸어요 ㅠㅠ,실습용 합성 텍스트,교육·실습용 자체 생성,"실제 개인정보 미사용, 가상값은 마스킹",토큰화 결과가 생각보다 복잡해서 조금 헷갈렸어요 ㅠㅠ,특이 패턴 없음,토큰화 결과가 생각보다 복잡해서 조금 헷갈렸어요 ㅠㅠ,0,0,"[토큰, 결과, 생각]","[토큰, 결과, 생각]","[(토큰, NNG), (결과, NNG), (생각, NNG), (하, XSA), (어..."
2,과제 예제는 유용하지만 실행 시간이 너무 길어요.,실습용 합성 텍스트,교육·실습용 자체 생성,"실제 개인정보 미사용, 가상값은 마스킹",과제 예제는 유용하지만 실행 시간이 너무 길어요.,특이 패턴 없음,과제 예제는 유용하지만 실행 시간이 너무 길어요.,0,0,"[과제, 예제, 유용, 실행, 시간]","[과제, 예제, 유용, 실행, 시간]","[(과제, NNG), (예제, NNG), (유용, NNG), (하, XSA), (지..."
3,새 모델 성능이 좋아서 다음 실험도 기대됩니다 😊,실습용 합성 텍스트,교육·실습용 자체 생성,"실제 개인정보 미사용, 가상값은 마스킹",새 모델 성능이 좋아서 다음 실험도 기대됩니다 😊,특이 패턴 없음,새 모델 성능이 좋아서 다음 실험도 기대됩니다 😊,1,0,"[모델, 성능, 다음, 실험, 기대]","[모델, 성능, 다음, 실험, 기대]","[(모델, NNG), (성능, NNG), (좋, VA), (어서, EC), (다음,..."
4,오류 메시지가 계속 나와서 정말 답답했어요...,실습용 합성 텍스트,교육·실습용 자체 생성,"실제 개인정보 미사용, 가상값은 마스킹",오류 메시지가 계속 나와서 정말 답답했어요...,반복 문자,오류 메시지가 계속 나와서 정말 답답했어요...,0,1,"[오류, 메시지]","[오류, 메시지]","[(오류, NNG), (메시지, NNG), (계속, MAG), (나오, VV), (..."
5,문의 주소는 student@example.com 같은 가상값으로만 테스트합니다.,실습용 합성 텍스트,교육·실습용 자체 생성,"실제 개인정보 미사용, 가상값은 마스킹",문의 주소는 student@example.com 같은 가상값으로만 테스트합니다.,이메일 가능성,문의 주소는 [EMAIL] 같은 가상값으로만 테스트합니다.,0,0,"[문의, 주소, 가상, 값, 테스트]","[문의, 주소, 가상, 값, 테스트]","[(문의, NNG), (주소, NNG), (같, VA), (가상, NNG), (값,..."


## 다음 주차 예고

다음 주차에는 전처리된 텍스트를 컴퓨터가 계산할 수 있는 수치 벡터로 표현합니다. 단어 빈도, Bag-of-Words, TF-IDF를 이용해 문서를 비교하고 분류 모델의 입력 특징을 구성합니다.